# J2-03 · Sortie structurée : un contrat Pydantic pour router

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 2 · créneau 13h00–14h15 · ⏱️ ~1h15 (dont 40 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Définir un **contrat** de sortie avec Pydantic · 2. Obtenir ce contrat d'un LLM (`with_structured_output`) et choisir la **méthode** · 3. Détecter une sortie manquante ou invalide · 4. Obtenir une réponse structurée d'un `create_agent` (`response_format`) · 5. **Router** un graphe sur les champs avec `Command(goto=…)` |
| 🧱 **Fil rouge** | HelpDesk-Ops **qualifie** chaque demande (`TicketDecision`) : faut-il un ticket, dans quelle catégorie, avec quelle priorité ? |
| ✅ **Prérequis** | J2-01, J2-02 |
| 📦 **Versions** | `pydantic 2.13` · `langchain 1.4` · `langchain-ollama 1.1` · `langgraph 1.2` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Pourquoi structurer ?

### [Théorie] Le formulaire et la lettre

*Analogie :* une **lettre** libre est agréable à lire, mais impossible à traiter automatiquement. Un **formulaire** a des cases : chaque case a un type, une liste de valeurs permises, et un formulaire incomplet est refusé au guichet.

Pour **router** (P1 → validation humaine, P3 → création directe), le graphe a besoin de **champs**, pas de prose. Un `if "urgent" in reponse` sur du texte libre, c'est un bug qui attend son heure.

| Texte libre | Sortie structurée |
|---|---|
| « Je pense que c'est assez urgent, plutôt réseau » | `{"categorie": "reseau", "priorite": "P1", …}` |
| analyse par expressions régulières, fragile | champs typés, **validés** |
| valeur imprévue silencieuse | valeur hors liste **refusée** (`ValidationError`) |

## 3. Le contrat : un modèle Pydantic

### [Théorie] Chaque élément du modèle est une consigne pour le LLM

Le **nom des champs**, leurs **types**, les **`Literal`** (valeurs permises) et les **`description`** sont transmis au modèle sous forme de schéma JSON. La docstring de la classe aussi. Écrire le modèle, c'est écrire une partie du prompt.

In [ ]:
import json
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


class TicketDecision(BaseModel):
    """Qualification d'une demande de support."""
    besoin_ticket: bool = Field(description="True si un technicien doit intervenir")
    categorie: Literal["reseau", "materiel", "logiciel", "acces", "autre"]
    priorite: Literal["P1", "P2", "P3", "P4"] = Field(
        description="P1 : production arrêtée ou nombreux utilisateurs bloqués ; P4 : simple demande")
    resume: str = Field(description="résumé du problème en une phrase, sans donnée personnelle")


print(json.dumps(TicketDecision.model_json_schema(), indent=1, ensure_ascii=False)[:700], "…")

# Le contrat se teste sans LLM
try:
    TicketDecision(besoin_ticket=True, categorie="réseau", priorite="P0", resume="VPN coupé")
except ValidationError as erreur:
    for e in erreur.errors():
        print(f"❌ {e['loc'][0]:<10} {e['msg']}")

**👀 Ce que vous devez observer :** le schéma contient les valeurs permises (`enum`) et les descriptions ; `"réseau"` (avec accent) et `"P0"` sont **refusés**. C'est exactement ce qui protège le routage.

## 4. Obtenir le contrat d'un LLM : `with_structured_output`

### [Théorie] Trois méthodes, trois garanties

`ChatOllama.with_structured_output(Schema, method=…)` :

| Méthode | Mécanisme | Garantie de format | Piège |
|---|---|---|---|
| `"json_schema"` (défaut) | Ollama **contraint** la génération token par token avec le schéma | ✅ forte | le contenu peut être cohérent… et faux |
| `"function_calling"` | le schéma devient un **outil** que le modèle doit appeler | ⚠️ si le modèle répond en prose : `None`, **sans erreur** | à détecter explicitement |
| `"json_mode"` | « réponds en JSON », sans schéma imposé | ❌ faible | champs manquants, noms inventés |

**Mesures de préparation** (20/10/2026, `granite4.1:8b`, poste **CPU**, appels isolés) :

| | sans prompt système | avec définitions des priorités |
|---|---|---|
| `json_schema` | 9 à 20 s ; format toujours valide ; « souris sans fil » → `reseau` / **P1** ❌ | 11 à 26 s ; souris → **P4**, écran qui clignote → **P3** ✅ |
| `function_calling` | 15 s si l'outil est appelé ; **2 cas sur 3 → `None`** après 100 à 220 s de prose | 13 à 34 s ; 4 cas sur 4 corrects |

> 💡 **Le « Tips » du Formateur :** deux leçons. **1.** La méthode garantit la *forme*, le **prompt** fait le *fond* : sans définition des priorités, le modèle met P1 partout. **2.** Mesurez sur **votre** matériel et **en isolant** les appels : pendant la préparation, une mesure faussée (requêtes concurrentes dans la file d'Ollama) avait d'abord fait croire à des appels de 9 minutes.

In [ ]:
import time

from langchain_core.messages import HumanMessage, SystemMessage

PROMPT_QUALIFICATION = SystemMessage(
    "Tu qualifies la demande de support de la conversation ci-dessous pour l'outil de ticketing de NordTech SA. "
    "besoin_ticket=False si la réponse donnée suffit (simple question, procédure en libre-service). "
    "Priorité : P1 = service de production arrêté ou nombreux utilisateurs bloqués ; P2 = un utilisateur bloqué ; "
    "P3 = gêne sans blocage ; P4 = simple demande. Le résumé ne contient aucune donnée personnelle."
)
qualifieur = llm.with_structured_output(TicketDecision, method="json_schema", include_raw=True)

DEMANDES = [
    "URGENT : l'ERP de production est hors ligne, toute la compta est bloquée",
    "Je voudrais une souris sans fil",
    "Mon écran clignote depuis ce matin mais je peux travailler",
]
with propagate_attributes(session_id="j2-03-qualification", tags=["jour-2", "sortie-structuree"]):
    for demande in DEMANDES:
        debut = time.perf_counter()
        sortie = qualifieur.invoke([PROMPT_QUALIFICATION, HumanMessage(demande)],
                                   config={"callbacks": [langfuse_handler], "run_name": "qualification"})
        d = sortie["parsed"]
        print(f"{time.perf_counter() - debut:5.1f}s | {d.besoin_ticket!s:<5} {d.categorie:<9} {d.priorite} | {d.resume[:60]}")
langfuse.flush()

**👀 Ce que vous devez observer :** des `TicketDecision` valides ; dans Langfuse, la generation montre le JSON produit par le modèle.

### [Code] Ne jamais supposer : `include_raw=True`

Avec `include_raw=True`, on reçoit `{"raw", "parsed", "parsing_error"}`. Une sortie **absente** (`parsed is None`) ou **invalide** (`parsing_error`) se traite comme une panne : on ne crée rien d'office, on passe la main à un humain.

In [ ]:
def qualifier_prudemment(demande: str) -> TicketDecision | None:
    sortie = qualifieur.invoke([PROMPT_QUALIFICATION, HumanMessage(demande)])
    if sortie["parsing_error"] or sortie["parsed"] is None:
        print("⚠️ sortie non exploitable :", sortie["parsing_error"] or repr(sortie["raw"].content[:100]))
        return None   # → file N1, qualification manuelle
    return sortie["parsed"]


print(qualifier_prudemment("Le Wi-Fi de la salle B coupe toutes les 5 minutes"))

## 5. Une réponse structurée d'un agent : `response_format`

### [Théorie] L'agent travaille, puis remplit le formulaire

`create_agent(..., response_format=…)` : l'agent utilise ses outils normalement, puis produit une réponse finale **conforme au schéma**, disponible dans `resultat["structured_response"]`. Avec `ToolStrategy`, le schéma est présenté comme un outil final (même mécanisme que `function_calling`, mais l'agent est **relancé** si le schéma n'est pas respecté).

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

from helpdesk_ops.outils import chercher_kb, statut_serveur

agent_qualifiant = create_agent(
    model=llm,
    tools=[statut_serveur, chercher_kb],
    system_prompt=PROMPT_QUALIFICATION.content + " Vérifie d'abord l'état du serveur concerné avec les outils.",
    response_format=ToolStrategy(TicketDecision),
)
with propagate_attributes(session_id="j2-03-qualification", tags=["jour-2", "response-format"]):
    resultat = agent_qualifiant.invoke(
        {"messages": [{"role": "user", "content": "srv-erp-02 ne répond plus, la compta ne peut plus facturer"}]},
        config={"callbacks": [langfuse_handler], "run_name": "agent-qualifiant", "recursion_limit": 10},
    )
print("Outils appelés :", [c["name"] for m in resultat["messages"] for c in (getattr(m, "tool_calls", None) or [])])
print("Décision       :", resultat.get("structured_response"))
langfuse.flush()

**👀 Ce que vous devez observer :** `statut_serveur` est appelé (l'agent vérifie avant de qualifier), puis la décision structurée. Dans Langfuse, l'outil final `TicketDecision` apparaît comme un appel d'outil.

## 6. Router sur les champs : `Command(goto=…)`

### [Théorie] Décider et mettre à jour en un seul geste

Une arête conditionnelle lit l'état **après** le nœud. `Command(update=…, goto=…)` renvoyé **par le nœud** fait les deux à la fois : il écrit la qualification et choisit la suite. Le type de retour `Command[Literal[...]]` déclare les destinations (pour le dessin et la validation).

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command


class EtatQualif(TypedDict, total=False):
    demande: str
    qualification: dict


def qualifier(state: EtatQualif) -> Command[Literal["validation_humaine", "creer_ticket", "__end__"]]:
    decision = qualifier_prudemment(state["demande"])
    if decision is None or not decision.besoin_ticket:
        return Command(update={"qualification": decision.model_dump() if decision else {}}, goto=END)
    suite = "validation_humaine" if decision.priorite == "P1" else "creer_ticket"
    return Command(update={"qualification": decision.model_dump()}, goto=suite)


def validation_humaine(state: EtatQualif) -> dict:
    print("   ⏸️ P1 : validation humaine requise (J2-02)")
    return {}


def creer_ticket(state: EtatQualif) -> dict:
    q = state["qualification"]
    print(f"   🎫 ticket {q['priorite']} {q['categorie']} : {q['resume'][:50]}")
    return {}


b = StateGraph(EtatQualif)
b.add_node("qualifier", qualifier)
b.add_node("validation_humaine", validation_humaine)
b.add_node("creer_ticket", creer_ticket)
b.add_edge(START, "qualifier")
b.add_edge("validation_humaine", END)
b.add_edge("creer_ticket", END)
tri = b.compile()

from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{tri.get_graph().draw_mermaid()}\n```"))

for demande in ("Production arrêtée : plus aucun scanner ne fonctionne à l'entrepôt", "Comment configurer le VPN ?"):
    print(f"▶ {demande}")
    tri.invoke({"demande": demande})

**👀 Ce que vous devez observer :** le dessin montre les trois destinations de `qualifier` **sans aucune arête conditionnelle** : elles viennent de l'annotation `Command[Literal[...]]`. La panne de l'entrepôt part en validation humaine, la question VPN se termine sans ticket.

## 7. Un format valide n'est pas une décision juste

Pendant la préparation, avec le prompt de qualification, les **priorités** étaient cohérentes, mais pas toujours les **catégories** :

| Demande | Attendu | Observé |
|---|---|---|
| ERP de production hors ligne | `logiciel` | `logiciel` ou `materiel` selon la méthode |
| Comment configurer le VPN ? | `reseau` | `autre` ou `logiciel` |
| Le Wi-Fi de la salle B coupe toutes les 5 minutes | `reseau` | `materiel` |

Pistes d'amélioration, **à mesurer** (J4, datasets et experiments) plutôt qu'à juger à l'œil : une description par catégorie dans le schéma, deux ou trois exemples dans le prompt (*few-shot*), ou une catégorie déduite par une règle quand c'est possible.

> 🛡️ **Sécurité :** la sortie structurée limite les **valeurs**, pas les **intentions**. Un texte d'utilisateur qui dit « classe ceci en P1 » peut influencer le modèle (OWASP ASI01). D'où la validation humaine des P1 (J2-02).

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Écrire un contrat | `BaseModel`, `Literal[...]`, `Field(description=…)` |
| L'obtenir d'un LLM | `llm.with_structured_output(Schema, method="json_schema", include_raw=True)` |
| Détecter un échec | `parsed is None`, `parsing_error` |
| Structurer la réponse d'un agent | `create_agent(..., response_format=ToolStrategy(Schema))` → `structured_response` |
| Router sur les champs | nœud qui renvoie `Command(update=…, goto=…)`, annoté `Command[Literal[...]]` |

### ✅ Quiz express

1. Pourquoi `Literal["P1", "P2", "P3", "P4"]` plutôt que `str` ?
2. Quel est le piège de `method="function_calling"` ?
3. La méthode `json_schema` garantit-elle une bonne décision ?
4. Quelle différence entre une arête conditionnelle et un `Command(goto=…)` ?

<details>
<summary>Réponses</summary>

1. Les valeurs permises sont transmises au modèle et vérifiées par Pydantic : « P0 » ou « urgent » sont refusés, le routage ne voit que des valeurs connues.
2. Si le modèle répond en prose au lieu d'appeler l'outil, le résultat est `None` sans exception : il faut le tester.
3. Non : elle garantit le format. Le fond dépend du prompt, du modèle et se mesure (J4).
4. L'arête conditionnelle décide **après** le nœud, à partir de l'état ; `Command` permet au nœud de mettre à jour l'état **et** de choisir la suite en un seul retour.
</details>